In [1]:
import cv2 
import urllib.request
import os
import time

In [ ]:
url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
filename = "haarcascade_frontalface_default.xml"

if not os.path.exists(filename):
    print('downloading haar cascade file')
    urllib.request.urlretrieve(url, filename)
    print('downloading completed')


save_path = r"Put Your Path In Order To Save"
os.makedirs(save_path, exist_ok=True)


last_capture_time = 0
capture_delay = 2  
face_count = 0    

def draw_box(img, classifier, scaleFactor, minNeighbors, color, text):
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    features = classifier.detectMultiScale(gray_img, scaleFactor, minNeighbors)
    
    detected = False

    for (x, y, w, h) in features:
        detected = True
        cv2.rectangle(img, (x,y), (x+w, y+h), color, 2)
        cv2.putText(img, text, (x, y-4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 1, cv2.LINE_AA)

    return img, detected

def detect(img, faceCascade):
    color = {"blue":(255,0,0)}
    img, detected = draw_box(img, faceCascade, 1.1, 10, color['blue'], "Face")
    return img, detected


faceCascade = cv2.CascadeClassifier(filename)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break

    img, detected = detect(frame, faceCascade)

    if detected:
        current_time = time.time()

        if current_time - last_capture_time > capture_delay:
            face_count += 1

            file_name = f"{save_path}/face_{face_count}.jpg"
            cv2.imwrite(file_name, frame)

            print(f"Saved: {file_name}")

            last_capture_time = current_time

    cv2.putText(img, f"Face Count: {face_count}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow("Face Detection", img)

    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()